# 生存分析模型比较实验 - 两侧CSA
## 36组数据 × 4种方法（Two-sided Conformalized Survival Analysis）

In [1]:
import numpy as np
import pandas as pd
np.random.seed(2026)

import warnings
warnings.filterwarnings('ignore')
import importlib
import config
importlib.reload(config)
from config import *
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = ['Heiti TC']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
# 生成36组数据
random_seed = 2026
sample_sizes = [100, 500, 2000]
cens_lambdas = [0.1, 0.3, 0.5]
p_list = [1, 50]
hetero_list = [False, True]

datasets = []
configs = []

for n in sample_sizes:
    for cens in cens_lambdas:
        for p in p_list:
            for hetero in hetero_list:
                X, surv, time, event, real_censor = generate_weibull_data(
                    n=n, p=p, hetero=hetero, cens_rate=cens
                )
                datasets.append((X, surv, time, event))
                configs.append({
                    'n': n, 'cens_rate': cens, 'p': p, 
                    'hetero': hetero, 'real_censor': real_censor
                })

print(f'✅ 生成完成：{len(datasets)}组数据')

✅ 生成完成：36组数据


In [3]:
# 批量拟合四种模型
results = []

for i, (X, surv, time, event) in enumerate(datasets):
    cfg = configs[i]
    print(f"第 {i+1} 组: n={cfg['n']}, p={cfg['p']}, cens={cfg['cens_rate']}, hetero={cfg['hetero']}")

    # 1. Kaplan-Meier
    kmf = fit_kaplan_meier(time, event)
    
    # 2. Cox
    cox = fit_cox(X, time, event)
    c_cox = evaluate_model(cox, X, time, event, 'cox')
    
    # 3. Weibull
    weibull = fit_weibull(X, time, event)
    c_weibull = evaluate_model(weibull, X, time, event, 'weibull')
    
    # 4. 随机生存森林
    rsf = fit_rsf(X, time, event)
    c_rsf = evaluate_model(rsf, X, time, event, 'rsf')
    
    results.append({
        **cfg,
        'C_Cox': c_cox,
        'C_Weibull': c_weibull,
        'C_RSF': c_rsf
    })
print("数据处理完成")
df_results = pd.DataFrame(results)
df_results

第 1 组: n=100, p=1, cens=0.1, hetero=False
第 2 组: n=100, p=1, cens=0.1, hetero=True
第 3 组: n=100, p=50, cens=0.1, hetero=False
第 4 组: n=100, p=50, cens=0.1, hetero=True
第 5 组: n=100, p=1, cens=0.3, hetero=False
第 6 组: n=100, p=1, cens=0.3, hetero=True
第 7 组: n=100, p=50, cens=0.3, hetero=False
第 8 组: n=100, p=50, cens=0.3, hetero=True
第 9 组: n=100, p=1, cens=0.5, hetero=False
第 10 组: n=100, p=1, cens=0.5, hetero=True
第 11 组: n=100, p=50, cens=0.5, hetero=False
第 12 组: n=100, p=50, cens=0.5, hetero=True
第 13 组: n=500, p=1, cens=0.1, hetero=False
第 14 组: n=500, p=1, cens=0.1, hetero=True
第 15 组: n=500, p=50, cens=0.1, hetero=False
第 16 组: n=500, p=50, cens=0.1, hetero=True
第 17 组: n=500, p=1, cens=0.3, hetero=False
第 18 组: n=500, p=1, cens=0.3, hetero=True
第 19 组: n=500, p=50, cens=0.3, hetero=False
第 20 组: n=500, p=50, cens=0.3, hetero=True
第 21 组: n=500, p=1, cens=0.5, hetero=False
第 22 组: n=500, p=1, cens=0.5, hetero=True
第 23 组: n=500, p=50, cens=0.5, hetero=False
第 24 组: n=500, p=50,

,n,cens_rate,p,hetero,real_censor,C_Cox,C_Weibull,C_RSF
0,100,0.1,1,False,0.1000,0.446192,0.553808,0.736801
1,100,0.1,1,True,0.1500,0.500457,0.499543,0.758455
2,100,0.1,50,False,0.1000,0.125339,0.872576,0.946611
3,100,0.1,50,True,0.1400,0.139631,0.831705,0.952443
4,100,0.3,1,False,0.3600,0.431607,0.568393,0.757503
5,100,0.3,1,True,0.3000,0.439537,0.560463,0.724106
6,100,0.3,50,False,0.2800,0.130685,0.865486,0.950455
7,100,0.3,50,True,0.3500,0.079135,0.916196,0.958466
8,100,0.5,1,False,0.4800,0.443014,0.556986,0.814286
9,100,0.5,1,True,0.3400,0.489948,0.510052,0.733958


In [4]:
# 保存结果
df_results.to_csv('模型拟合结果.csv', index=False)
print('✅ 结果已保存')

✅ 结果已保存


In [5]:

csa_results = []

# 基础模型选择：KM、Cox、Weibull、RSF
base_models = ['km', 'cox', 'weibull', 'rsf']
alpha = 0.1

for i, (X, surv, time, event) in enumerate(datasets):
    cfg = configs[i]

    print(f"CSA 第 {i+1} 组: n={cfg['n']}, p={cfg['p']}, cens={cfg['cens_rate']}, hetero={cfg['hetero']}")
    X_train, time_train, event_train, X_cal, time_cal, event_cal, X_test, time_test, event_test = split_survival_data(
        X, time, event, test_size=0.2, cal_size=0.25, random_state=2026
    )

    kmf = fit_kaplan_meier(time_train, event_train)
    cox = fit_cox(X_train, time_train, event_train)
    weibull = fit_weibull(X_train, time_train, event_train)
    rsf = fit_rsf(X_train, time_train, event_train)

    model_map = {
        'km': kmf,
        'cox': cox,
        'weibull': weibull,
        'rsf': rsf
    }

    for model_type in base_models:
        model = model_map[model_type]
        
        # 使用两侧CSA方法（完全遵循论文标准）
        lower, upper, q_value, classification = fit_csa_intervals_two_sided(
            model, X_train, time_train, event_train,
            X_cal, time_cal, event_cal, X_test,
            alpha=alpha, model_type=model_type
        )
        
        # data_type='synthetic' 表示所有样本有真实T，采用统一判定规则
        metrics = evaluate_interval_coverage_two_sided(
            lower, upper, time_test, event_test, classification,
            data_type='synthetic'
        )

        csa_results.append({
            'n': cfg['n'],
            'p': cfg['p'],
            'cens_rate': cfg['cens_rate'],
            'hetero': cfg['hetero'],
            'model_type': model_type,
            'method': 'Two-sided CSA (论文标准-Conformal p-value)',
            'alpha': alpha,
            'coverage': metrics['coverage'],
            'coverage_two_sided': metrics['coverage_two_sided'],
            'coverage_one_sided': metrics['coverage_one_sided'],
            'mean_width_two_sided': metrics['mean_width_two_sided'],
            'std_width_two_sided': metrics['std_width_two_sided'],
            'q_value': q_value,
            'num_two_sided': metrics['num_two_sided'],
            'num_one_sided': metrics['num_one_sided']
        })


# 保存两侧CSA实验结果
df_csa = pd.DataFrame(csa_results)
df_csa.to_csv('Two_sided_CSA_results.csv', index=False)
print('两侧CSA实验完成（论文标准方法-合成数据）：', df_csa.shape[0], '条记录')

# 简要汇总
print('\n✓ 两侧CSA汇总（论文标准实现-合成数据）：按模型类型查看平均指标')
print(df_csa.groupby('model_type')[['coverage','coverage_two_sided','coverage_one_sided','mean_width_two_sided']].mean())
print('\n样本分布情况（两侧 vs 单侧）：')
print(df_csa.groupby('model_type')[['num_two_sided','num_one_sided']].mean())

CSA 第 1 组: n=100, p=1, cens=0.1, hetero=False
CSA 第 2 组: n=100, p=1, cens=0.1, hetero=True
CSA 第 3 组: n=100, p=50, cens=0.1, hetero=False
CSA 第 4 组: n=100, p=50, cens=0.1, hetero=True
CSA 第 5 组: n=100, p=1, cens=0.3, hetero=False
CSA 第 6 组: n=100, p=1, cens=0.3, hetero=True
CSA 第 7 组: n=100, p=50, cens=0.3, hetero=False
CSA 第 8 组: n=100, p=50, cens=0.3, hetero=True
CSA 第 9 组: n=100, p=1, cens=0.5, hetero=False
CSA 第 10 组: n=100, p=1, cens=0.5, hetero=True
CSA 第 11 组: n=100, p=50, cens=0.5, hetero=False
CSA 第 12 组: n=100, p=50, cens=0.5, hetero=True
CSA 第 13 组: n=500, p=1, cens=0.1, hetero=False
CSA 第 14 组: n=500, p=1, cens=0.1, hetero=True
CSA 第 15 组: n=500, p=50, cens=0.1, hetero=False
CSA 第 16 组: n=500, p=50, cens=0.1, hetero=True
CSA 第 17 组: n=500, p=1, cens=0.3, hetero=False
CSA 第 18 组: n=500, p=1, cens=0.3, hetero=True
CSA 第 19 组: n=500, p=50, cens=0.3, hetero=False
CSA 第 20 组: n=500, p=50, cens=0.3, hetero=True
CSA 第 21 组: n=500, p=1, cens=0.5, hetero=False
CSA 第 22 组: n=500, p=1

## 两侧CSA方法说明

### 方法原理

两侧CSA采用分类策略，区分未删失和删失样本：

1. **删失状态分类**：使用随机森林分类器预测P(Δ=1|X)，识别哪些样本的生存时间被完全观测

2. **混合区间生成**：
   - 预测为**未删失(Δ=1)**的样本：构造两侧区间 [L, U]
   - 预测为**删失(Δ=0)**的样本：构造单侧区间 [L, ∞)
   
3. **覆盖率保证**（有限样本）：
   - 对两侧区间分配 α/2
   - 对单侧区间分配 α/2
   - 整体：P(T ∉ Ĉ(X)) ≤ α

### 关键特性

| 特性 | 描述 |
|------|------|
| **上界计算** | 通过反演生存函数 F̂⁻¹(0.5 + q) 得到，超出支撑时为∞ |
| **避免权重** | 无需估计删失机制 ĉ(x)，规避数值不稳定性 |
| **有限样本** | 基于conformal p-value的分类，提供有限样本保证 |
| **混合覆盖** | 利用未删失样本的完整信息提供更精确的两侧区间 |

### 与传统CSA的比较

| 维度 | 传统CSA | 两侧CSA |
|------|----------|----------|
| **区间形式** | 所有[L, ∞) | 混合：两侧 + 单侧 |
| **权重估计** | 需要 | 不需要 |
| **上界** | 无 | 有限 |
| **覆盖类型** | 统一 | 分群体 |
| **理论样本量** | 渐近 | **有限样本** |
| **数值稳定性** | 中 | 好 |

### 实验输出

- **coverage**: 总体覆盖率
- **coverage_two_sided**: 两侧区间的覆盖率（应≥ α/2）
- **coverage_one_sided**: 单侧区间的覆盖率（应≥ α/2）
- **mean_width_two_sided**: 两侧区间的平均宽度
- **num_two_sided**: 获得两侧区间的样本数
- **num_one_sided**: 获得单侧区间的样本数